# Test No.1: Training and testing the newly refactored Classifer

## Imports for test

In [1]:
from src.classifiers.trainer.training_script import train_and_report_all_models

import os
import torch
import matplotlib.pyplot as plt
import numpy as np
from src.classifiers.ex_classifier import load_trained_model, CelebAClassifierDataset
from src.classifiers.cam_utils import get_gradcam_per_attribute
from src.classifiers.cam_utils import get_gradcam_mask
from src.classifiers.ig_utils import get_ig_safe
from src.classifiers.visualizations import visualize_per_attribute_saliency, visualize_attribute_comparison
import torchvision.transforms as T
import cv2
import pandas as pd

## Training and saliency wrapper functions here

In [5]:
def training():
    class Args:
        pass
    args = Args()
    args.dataset_dir = 'Dataset/celeba_70percent_721'
    
    # === SPEED OPTIMIZATION ===
    args.epochs = 30                    # Reduced from 30
    args.batch_size = 128              # Increased from 128 (requires ~16GB GPU)
    # For smaller GPU (8GB), use: args.batch_size = 64
    
    args.lr = 2e-3                      # Faster convergence
    args.save_dir = 'outputs/trained_classifiers(0512)'
    args.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    args.num_workers = 6               # Parallel data loading
    args.use_amp = False               # Mixed precision training
    
    attribute_names = [
    # --- Global / Structure ---
    'Male',
    'Young',
    'Bald',
    'Chubby',
    
    # --- Hair Texture & Color ---
    'Black_Hair',
    'Blond_Hair',
    'Brown_Hair',
    'Gray_Hair',
    'Receding_Hairline',
    'Bangs',
    'Straight_Hair',
    'Wavy_Hair',
    
    # --- Facial Features (Geometry) ---
    'Arched_Eyebrows',
    'Bushy_Eyebrows',
    'Bags_Under_Eyes',
    'Narrow_Eyes',
    'Big_Nose',
    'Pointy_Nose',
    'High_Cheekbones',
    'Rosy_Cheeks',
    'Big_Lips',
    'Mouth_Slightly_Open',
    'Smiling',
    
    # --- Facial Hair ---
    'Mustache',
    'Goatee',
    'No_Beard',
    
    # --- Accessories & Details ---
    'Eyeglasses',
    'Wearing_Hat',
    'Wearing_Earrings',
    'Wearing_Necktie',
    'Heavy_Makeup',
    'Wearing_Lipstick'
]

    train_and_report_all_models(args, attribute_names)


def test_models_saliency_subset(model_path, test_img_dir, attr_file, attribute_names, subset_filenames, device=None, root_out="outputs/saliency_test"):
    """
    Test a single ExplainableClassifier model on subset of images with codebook-style visualizations.
    Save per-attribute saliency figures to:
        saliency_test/img_no.{idx}/attribute:{attr_name}/
    
    Args:
        model_path: Path to trained model .pth file
        test_img_dir: Directory containing test images
        attr_file: Path to attribute CSV file
        attribute_names: List of attribute names (must match model training attributes)
        subset_filenames: List of image filenames to test
        device: torch.device or None
        root_out: Output root directory
    """
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    os.makedirs(root_out, exist_ok=True)

    # Load all attributes from CSV
    df = pd.read_csv(attr_file)

    resize_transform = T.Compose([
        T.ToPILImage(),
        T.Resize(128),
        T.CenterCrop(128),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    # Load model once
    if not os.path.exists(model_path):
        print(f"❌ Model not found: {model_path}")
        return
    
    print(f"Loading model from: {model_path}")
    model = load_trained_model(model_path, num_classes=len(attribute_names), 
                              attribute_names=attribute_names, device=device)
    model.eval()

    for img_idx, img_name in enumerate(subset_filenames):
        img_path = os.path.join(test_img_dir, img_name)
        if not os.path.exists(img_path):
            print(f"⚠️  Image not found: {img_path}")
            continue

        img = plt.imread(img_path)
        if img.max() > 1.0:
            img = img / 255.0

        # Preprocess image for model
        img_tensor_input = (img * 255).astype(np.uint8)
        img_tensor = resize_transform(img_tensor_input)
        img_tensor = img_tensor.unsqueeze(0).float().to(device)

        # For plotting (denormalize back to [0,1])
        img_disp = img_tensor.squeeze().cpu().permute(1, 2, 0).numpy()
        img_disp = img_disp * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        img_disp = np.clip(img_disp, 0, 1)

        # For overlays (uint8 BGR)
        img_disp_uint8 = (img_disp * 255).astype(np.uint8)
        if img_disp_uint8.ndim == 2:
            base_img_uint8 = cv2.cvtColor(img_disp_uint8, cv2.COLOR_GRAY2BGR)
        elif img_disp_uint8.shape[2] == 1:
            base_img_uint8 = cv2.cvtColor(img_disp_uint8, cv2.COLOR_GRAY2BGR)
        else:
            base_img_uint8 = img_disp_uint8.copy()

        # Get label row and extract attributes
        row = df[df[df.columns[0]] == img_name].iloc[0]
        label_vec_full = []
        for attr in attribute_names:
            if attr in row.index:
                label_vec_full.append(row[attr])
            else:
                label_vec_full.append(0)
        
        label_vec = torch.tensor(label_vec_full, dtype=torch.float32).unsqueeze(0).to(device)

        print(f"\n✓ Processing image {img_idx+1}/{len(subset_filenames)}: {img_name}")
        img_out_root = os.path.join(root_out, f"img_no.{img_idx}")

        with torch.no_grad():
            logits = model(img_tensor)
            probs = torch.sigmoid(logits)

        # Get per-attribute CAMs and IGs (GradCAM++ codebook style)
        cam_maps = get_gradcam_per_attribute(model, img_tensor, device=device)  # [B, num_classes, H, W]
        ig_mask = get_ig_safe(model, img_tensor, label_vec, spatial=True)        # [B, num_classes, H, W]

        # === Visualization 1: All attributes per-image ===
        fig1 = visualize_per_attribute_saliency(model, img_tensor, cam_maps, ig_mask, 
                                               probs, attribute_names, device=device)
        all_attr_path = os.path.join(img_out_root, 'all_attributes_saliency.png')
        os.makedirs(img_out_root, exist_ok=True)
        fig1.savefig(all_attr_path, dpi=150, bbox_inches='tight')
        plt.close(fig1)
        print(f"   ✅ Saved: all_attributes_saliency.png")

        # === Visualization 2: Top-K comparison (GradCAM++ vs IG) ===
        fig2 = visualize_attribute_comparison(img_tensor, cam_maps, ig_mask, probs, 
                                             attribute_names, top_k=10, device=device)
        top_k_path = os.path.join(img_out_root, 'top_k_comparison.png')
        fig2.savefig(top_k_path, dpi=150, bbox_inches='tight')
        plt.close(fig2)
        print(f"   ✅ Saved: top_k_comparison.png")

        # === Visualization 3: Per-attribute detailed view ===
        cam_np = cam_maps.squeeze(0).cpu().numpy()  # [num_classes, H, W]
        ig_np = ig_mask.squeeze(0).cpu().numpy()     # [num_classes, H, W]
        pred_np = probs[0].cpu().numpy()

        for attr_idx, attr_name in enumerate(attribute_names):
            cam_attr = cam_np[attr_idx]
            ig_attr = ig_np[attr_idx]

            # Normalize
            cam_norm = (cam_attr - cam_attr.min()) / (cam_attr.max() - cam_attr.min() + 1e-8)
            ig_norm = (ig_attr - ig_attr.min()) / (ig_attr.max() - ig_attr.min() + 1e-8)

            # Create heatmaps
            cam_heat = cv2.applyColorMap((cam_norm * 255).astype(np.uint8), cv2.COLORMAP_JET)
            ig_heat = cv2.applyColorMap((ig_norm * 255).astype(np.uint8), cv2.COLORMAP_HOT)

            # Resize if needed
            if cam_heat.shape[:2] != base_img_uint8.shape[:2]:
                cam_heat = cv2.resize(cam_heat, (base_img_uint8.shape[1], base_img_uint8.shape[0]))
            if ig_heat.shape[:2] != base_img_uint8.shape[:2]:
                ig_heat = cv2.resize(ig_heat, (base_img_uint8.shape[1], base_img_uint8.shape[0]))

            # Create overlays
            cam_overlay = cv2.addWeighted(base_img_uint8, 0.5, cam_heat, 0.5, 0)
            ig_overlay = cv2.addWeighted(base_img_uint8, 0.5, ig_heat, 0.5, 0)
            
            # Combined overlay: blend both CAM and IG
            combined_overlay = cv2.addWeighted(cam_overlay, 0.5, ig_overlay, 0.5, 0)

            # Convert to RGB for matplotlib
            cam_overlay_rgb = cv2.cvtColor(cam_overlay, cv2.COLOR_BGR2RGB)
            ig_overlay_rgb = cv2.cvtColor(ig_overlay, cv2.COLOR_BGR2RGB)
            combined_overlay_rgb = cv2.cvtColor(combined_overlay, cv2.COLOR_BGR2RGB)
            img_disp_bgr = cv2.cvtColor((img_disp * 255).astype(np.uint8), cv2.COLOR_RGB2BGR)
            img_disp_rgb = cv2.cvtColor(img_disp_bgr, cv2.COLOR_BGR2RGB) / 255.0

            # Create per-attribute figure with 4 subplots
            fig, axs = plt.subplots(1, 4, figsize=(20, 5))
            
            # Subplot 0: Original image
            axs[0].imshow(img_disp)
            axs[0].set_title(f'Original Image\n{attr_name}', fontsize=12, fontweight='bold')
            axs[0].axis('off')
            
            # Subplot 1: CAM overlay
            axs[1].imshow(cam_overlay_rgb)
            axs[1].set_title(f'GradCAM++ Overlay\n(Pred: {pred_np[attr_idx]:.3f})', fontsize=12, fontweight='bold')
            axs[1].axis('off')
            
            # Subplot 2: IG overlay
            axs[2].imshow(ig_overlay_rgb)
            axs[2].set_title(f'IG Overlay\n(Pred: {pred_np[attr_idx]:.3f})', fontsize=12, fontweight='bold')
            axs[2].axis('off')
            
            # Subplot 3: Combined overlay (CAM + IG)
            axs[3].imshow(combined_overlay_rgb)
            axs[3].set_title(f'Combined (CAM + IG)\n(Pred: {pred_np[attr_idx]:.3f})', fontsize=12, fontweight='bold')
            axs[3].axis('off')
            
            plt.suptitle(f'{img_name} | {attr_name} | Attention Visualization', 
                        fontsize=14, fontweight='bold', y=0.98)
            plt.tight_layout()
            
            attr_dir = os.path.join(img_out_root, f"attribute:{attr_name}")
            os.makedirs(attr_dir, exist_ok=True)
            attr_path = os.path.join(attr_dir, 'saliency.png')
            fig.savefig(attr_path, dpi=150, bbox_inches='tight')
            plt.close(fig)

    print(f"\n✅ Saliency test complete. Results saved to: {root_out}")


## Classifier training

In [3]:
training()

Auto-detected paths:
  train_dir: Dataset/celeba_70percent_721/train/img_align_celeba
  val_dir: Dataset/celeba_70percent_721/val/img_align_celeba
  train_attr_file: Dataset/celeba_70percent_721/train/list_attr_celeba.csv
  val_attr_file: Dataset/celeba_70percent_721/val/list_attr_celeba.csv

Training ExplainableClassifier (MobileNetV3-Small backbone)
Batch Size: 128 | Epochs: 30 | Workers: 6

Training ExplainableClassifier (MobileNetV3-Small backbone)
Batch Size: 128 | Epochs: 30 | Workers: 6
Epoch 1/30 | Train Loss: 0.2465 | Val Loss: 0.2257 | Val Acc: 0.9008
Epoch 2/30 | Train Loss: 0.2103 | Val Loss: 0.2183 | Val Acc: 0.9031
Epoch 3/30 | Train Loss: 0.2026 | Val Loss: 0.2115 | Val Acc: 0.9062
Epoch 4/30 | Train Loss: 0.1981 | Val Loss: 0.2083 | Val Acc: 0.9082
Epoch 5/30 | Train Loss: 0.1943 | Val Loss: 0.2054 | Val Acc: 0.9093
Epoch 6/30 | Train Loss: 0.1912 | Val Loss: 0.2070 | Val Acc: 0.9088
Epoch 7/30 | Train Loss: 0.1879 | Val Loss: 0.2018 | Val Acc: 0.9109
Epoch 8/30 | Train

## Saliency test on the classifiers

In [6]:
# --- GradCAM++ and IG visualization (codebook style) ---
model_path = 'outputs/trained_classifiers(0512)/best_model.pth'
test_img_dir = 'Dataset/celeba_70percent_721/test/img_align_celeba'
attr_file = 'Dataset/celeba_70percent_721/test/list_attr_celeba.csv'
attribute_names = [
    # --- Global / Structure ---
    'Male',
    'Young',
    'Bald',
    'Chubby',
    
    # --- Hair Texture & Color ---
    'Black_Hair',
    'Blond_Hair',
    'Brown_Hair',
    'Gray_Hair',
    'Receding_Hairline',
    'Bangs',
    'Straight_Hair',
    'Wavy_Hair',
    
    # --- Facial Features (Geometry) ---
    'Arched_Eyebrows',
    'Bushy_Eyebrows',
    'Bags_Under_Eyes',
    'Narrow_Eyes',
    'Big_Nose',
    'Pointy_Nose',
    'High_Cheekbones',
    'Rosy_Cheeks',
    'Big_Lips',
    'Mouth_Slightly_Open',
    'Smiling',
    
    # --- Facial Hair ---
    'Mustache',
    'Goatee',
    'No_Beard',
    
    # --- Accessories & Details ---
    'Eyeglasses',
    'Wearing_Hat',
    'Wearing_Earrings',
    'Wearing_Necktie',
    'Heavy_Makeup',
    'Wearing_Lipstick'
]
subset_filenames = ['000007.jpg', '000018.jpg']

test_models_saliency_subset(model_path, test_img_dir, attr_file, attribute_names, subset_filenames)

Loading model from: outputs/trained_classifiers(0512)/best_model.pth

✓ Processing image 1/2: 000007.jpg

✓ Processing image 1/2: 000007.jpg


/mnt/a/Ubuntu/Lab_testing_n_research/condR3GAN/ceGAN/src/classifiers/visualizations.py:229: UserWarning: Tight layout not applied. tight_layout cannot make Axes height small enough to accommodate all Axes decorations.
  plt.tight_layout()


   ✅ Saved: all_attributes_saliency.png
   ✅ Saved: top_k_comparison.png
   ✅ Saved: top_k_comparison.png

✓ Processing image 2/2: 000018.jpg

✓ Processing image 2/2: 000018.jpg
   ✅ Saved: all_attributes_saliency.png
   ✅ Saved: all_attributes_saliency.png
   ✅ Saved: top_k_comparison.png
   ✅ Saved: top_k_comparison.png

✅ Saliency test complete. Results saved to: outputs/saliency_test

✅ Saliency test complete. Results saved to: outputs/saliency_test


# Test No.2: Training and testing the newly refactored vq_vae

### HR-VQGAN Training Setup
Initialize and configure the HR-VQGAN (Hierarchical Residual VQ-VAE with Discriminator) for unsupervised training on CelebA.

In [1]:
import torch
from src.unsupervised_latentspace.config import cfg
from src.unsupervised_latentspace.trainer import train

# ============================================================================
# HVQ-GAN Training V3 - Step-based Unsupervised Training
# ============================================================================

print("="*70)
print("🔧 HVQ-GAN TRAINING V3")
print("="*70)

# Configuration
cfg.total_steps = 30000        # Total training steps
cfg.batch_size = 16             # Batch size
cfg.save_interval = 2500        # Save checkpoint every 2500 steps
cfg.sample_interval = 1000     # ✅ INCREASE: Sample every 1000 steps (less frequent)
cfg.log_interval = 1           # Log every 1 step
cfg.num_samples = 8
cfg.device = "cuda" if torch.cuda.is_available() else "cpu"
cfg.data_path = "Dataset/celeba_70percent_721/train"
cfg.save_dir = "./outputs/checkpoints_production"
cfg.disc_start_step = 25000      # Start discriminator after 10000 steps warmup
cfg.use_amp = False             # ✅ DISABLE AMP initially (causing overflow)

print(f"\n📋 Configuration:")
print(f"   Device: {cfg.device}")
print(f"   Image Size: {cfg.image_size}x{cfg.image_size}")
print(f"   Batch Size: {cfg.batch_size}")
print(f"   Total Steps: {cfg.total_steps:,}")
print(f"   Log Interval: {cfg.log_interval} steps")
print(f"   Save Interval: {cfg.save_interval:,} steps")
print(f"   Sample Interval: {cfg.sample_interval:,} steps")
print(f"   Discriminator Start: Step {cfg.disc_start_step:,}")
print(f"   Learning Rate: {cfg.learning_rate}")
print(f"   Use AMP: {cfg.use_amp} (disabled for stability)")
print(f"   Loss Weights: {cfg.weights}")
print("="*70 + "\n")

# Verify data path
import os
if not os.path.exists(cfg.data_path):
    print(f"⚠️  WARNING: Data path not found: {cfg.data_path}")
    print(f"   Please update cfg.data_path to point to your CelebA dataset")
else:
    img_dir = os.path.join(cfg.data_path, "img_align_celeba")
    if os.path.exists(img_dir):
        num_images = len([f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png'))])
        print(f"✅ Found {num_images} training images")
    else:
        print(f"⚠️  Image directory not found: {img_dir}")

print("\n🚀 Starting step-based training...\n")

try:
    train()
    print("\n" + "="*70)
    print("✅ HVQ-GAN TRAINING COMPLETED!")
    print("="*70)
    print(f"\n📁 Checkpoints saved to: {cfg.save_dir}")
    print(f"📁 Samples saved to: {os.path.join(cfg.save_dir, 'samples')}")
    print(f"📁 Logs saved to: {os.path.join(cfg.save_dir, 'logs')}")
except KeyboardInterrupt:
    print("\n⏹️  Training interrupted by user")
except Exception as e:
    print(f"\n❌ Training error: {e}")
    import traceback
    traceback.print_exc()

🔧 HVQ-GAN TRAINING V3 (Step-based)

📋 Configuration:
   Device: cuda
   Image Size: 128x128
   Batch Size: 16
   Total Steps: 30,000
   Log Interval: 1 steps
   Save Interval: 2,500 steps
   Sample Interval: 1,000 steps
   Discriminator Start: Step 25,000
   Learning Rate: 0.0003
   Use AMP: False (disabled for stability)
   Loss Weights: {'recon': 1.0, 'vq': 1.0, 'perceptual': 1.0, 'disc': 0.8}

✅ Found 99272 training images

🚀 Starting step-based training...

🔧 HVQ-GAN TRAINING V3 (Step-based, Unsupervised)
Device: cuda
Total Steps: 30,000
Batch Size: 16
Log Interval: 1 steps
Save Interval: 2,500 steps
Sample Interval: 1,000 steps
📁 Output Directory: ./outputs/checkpoints_production
   ├── checkpoints/              (model checkpoints)
   ├── samples/                  (progressive decoding visualizations)
   ├── metrics_and_visualisations/ (training metrics & analysis)
   └── logs/                     (training logs)

Loading VGG for Perceptual Loss...


/mnt/a/Ubuntu/lab.venv/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/mnt/a/Ubuntu/lab.venv/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


✅ VGG loaded

Training images: 99272
Batches per cycle: 6205



Training:   8%|▊         | 2500/30000 [04:30<45:39, 10.04step/s, Tot=0.4043, Rec=0.0205, VQ=0.0243, Perc=0.3594, Ppl=158.72]  


📊 CREATING EVALUATION REPORT AT STEP 2,500
✅ Metrics comparison saved: metrics_comparison_step_2500.png
✅ Cascade hierarchy saved: cascade_hierarchy_step_2500.png

   Metrics history: metrics_history.csv

✅ Step 2,500 Checkpoint:
   Checkpoint: checkpoint_step_2500.pth
   Metrics: MSE=0.0205, PSNR=22.89 dB, SSIM=0.786
   Codebook: Top usage=0.4%, Mid usage=0.4%, Bottom usage=0.4%


...

Training: 100%|██████████| 30000/30000 [1:23:10<00:00,  6.16step/s, Tot=0.3880, Rec=0.0213, VQ=0.0294, Perc=0.3416, Ppl=210.45, DiscG=0.7961, DiscD=0.8607]   


📊 CREATING EVALUATION REPORT AT STEP 30,000
✅ Metrics comparison saved: metrics_comparison_step_30000.png


Training: 100%|██████████| 30000/30000 [1:25:29<00:00,  5.85step/s, Tot=0.3880, Rec=0.0213, VQ=0.0294, Perc=0.3416, Ppl=210.45, DiscG=0.7961, DiscD=0.8607]

✅ Cascade hierarchy saved: cascade_hierarchy_step_30000.png

   Metrics history: metrics_history.csv

✅ Step 30,000 Checkpoint:
   Checkpoint: checkpoint_step_30000.pth
   Metrics: MSE=0.0213, PSNR=22.73 dB, SSIM=0.846
   Codebook: Top usage=0.4%, Mid usage=0.4%, Bottom usage=0.4%

✅ HVQ-GAN TRAINING COMPLETED SUCCESSFULLY!

📁 Output Structure:
   ./outputs/checkpoints_production/
   ├── checkpoints/
   │   ├── checkpoint_step_5000.pth
   │   ├── checkpoint_step_10000.pth
   │   └── model_latest.pth
   ├── samples/
   │   ├── progressive_step_2500_comparison.png
   │   ├── progressive_step_5000_comparison.png
   │   └── progressive_step_10000_comparison.png
   ├── metrics_and_visualisations/
   │   ├── loss_curves_step_5000.png
   │   ├── metrics_comparison_step_5000.png
   │   ├── cascade_hierarchy_step_5000.png
   │   ├── codebook_usage_step_5000.png
   │   ├── codebook_analysis_step_5000.txt
   │   └── metrics_history.csv
   └── logs/
       └── training_log.txt

📊 Final step: 30,00


✅ HVQ-GAN TRAINING COMPLETED!

📁 Checkpoints saved to: ./outputs/checkpoints_production
📁 Samples saved to: ./outputs/checkpoints_production/samples
📁 Logs saved to: ./outputs/checkpoints_production/logs


# Test 3: Training Counterfactual Generator (ceGAN)
Initialize and train the comprehensive counterfactual generation model with logging, checkpointing, and visualization.

In [2]:
# Hyperparameter Initialization
from src.synthesis.config import Config

cfg = Config()
cfg.decoder_pretrain_epochs = 10
cfg.decoder_lr = 1e-3
cfg.decoder_checkpoint_dir = "outputs/synth_network/stylegan_decoder"
cfg.decoder_checkpoint_path = "outputs/synth_network/stylegan_decoder/latest.pth"
cfg.batch_size = 16
cfg.num_epochs = 20
cfg.accumulation_steps = 8
cfg.num_workers = 6
cfg.learning_rate = 1e-4
cfg.data_root = "Dataset/celeba_70percent_721/train"

# StyleGAN Decoder Pre-Training
from src.synthesis.dataset import get_loader
from src.synthesis.train_comprehensive import ComprehensiveTrainer

loader_train = get_loader(cfg, split='train', batch_size=cfg.batch_size, shuffle=True)
loader_val = get_loader(cfg, split='val', batch_size=cfg.batch_size, shuffle=False)

Loaded train split: 69490 images
Loaded val split: 9927 images


## Decoder Pre-training

In [ ]:
trainer = ComprehensiveTrainer(cfg, experiment_name="ceGAN_counterfactual")
trainer.recon_training(loader_train)

Loaded train split: 69490 images
Loaded val split: 9927 images
📁 Experiment directory: outputs/experiments/ceGAN_counterfactual_20251211_162329
⚠️ Decoder checkpoint not found at outputs/synth_network/stylegan_decoder/latest.pth; starting with random weights.
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


/mnt/a/Ubuntu/lab.venv/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/mnt/a/Ubuntu/lab.venv/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /mnt/a/Ubuntu/lab.venv/lib/python3.11/site-packages/lpips/weights/v0.1/vgg.pth
ℹ️ Decoder checkpoint not found at outputs/synth_network/stylegan_decoder/latest.pth; will train the decoder from scratch.
🎯 Starting StyleGAN decoder pre-training for 10 epoch(s)


[Decoder Pretrain] 0: 100%|██████████| 4343/4343 [13:12<00:00,  5.48it/s, L1=0.0551, LPIPS=0.135]


🖼️ Generating reconstruction preview...
🖼️ Saved decoder preview: outputs/synth_network/stylegan_decoder/previews/epoch_000.png
💾 Saved decoder checkpoints to outputs/synth_network/stylegan_decoder/decoder_pretrained.pth and outputs/synth_network/stylegan_decoder/latest.pth


...

🖼️ Generating reconstruction preview...
🖼️ Saved decoder preview: outputs/synth_network/stylegan_decoder/previews/epoch_007.png


[Decoder Pretrain] 8: 100%|██████████| 4343/4343 [12:43<00:00,  5.69it/s, L1=0.0432, LPIPS=0.103] 


🖼️ Generating reconstruction preview...
🖼️ Saved decoder preview: outputs/synth_network/stylegan_decoder/previews/epoch_008.png


[Decoder Pretrain] 9: 100%|██████████| 4343/4343 [13:18<00:00,  5.44it/s, L1=0.0504, LPIPS=0.105] 


🖼️ Generating reconstruction preview...
🖼️ Saved decoder preview: outputs/synth_network/stylegan_decoder/previews/epoch_009.png
✅ Decoder pre-training complete


## Decoder refinement training

In [2]:
# GAN refinement (decoder sharpening)
print("dataset at:", cfg.data_root)


from src.synthesis.train_sharpening import train_sharpening
train_sharpening(cfg.decoder_checkpoint_path, cfg=cfg)

dataset at: Dataset/celeba_70percent_721/train
🧭 Loaded StyleGAN decoder weights from outputs/synth_network/stylegan_decoder/latest.pth
🔄 Loading pre-trained decoder for sharpening: outputs/synth_network/stylegan_decoder/latest.pth
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


/mnt/a/Ubuntu/lab.venv/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/mnt/a/Ubuntu/lab.venv/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /mnt/a/Ubuntu/lab.venv/lib/python3.11/site-packages/lpips/weights/v0.1/vgg.pth
Loaded train split: 69490 images
Loaded val split: 9927 images
🚀 Starting adversarial sharpening phase...


Sharpening Epoch 0: 100%|██████████| 8686/8686 [41:46<00:00,  3.47it/s, D=1.62, G_Adv=0.118, L1=0.0509, LPIPS=0.124]    


💾 Saved EMA decoder weights to outputs/synth_network/stylegan_decoder_sharpened


...

Sharpening Epoch 16: 100%|██████████| 8686/8686 [21:11<00:00,  6.83it/s, D=0.552, G_Adv=0.871, L1=0.0518, LPIPS=0.141]   


💾 Saved EMA decoder weights to outputs/synth_network/stylegan_decoder_sharpened


Sharpening Epoch 18: 100%|██████████| 8686/8686 [21:09<00:00,  6.84it/s, D=0.521, G_Adv=0.866, L1=0.0525, LPIPS=0.14]    


💾 Saved EMA decoder weights to outputs/synth_network/stylegan_decoder_sharpened


Sharpening Epoch 19: 100%|██████████| 8686/8686 [21:13<00:00,  6.82it/s, D=0.0641, G_Adv=1.05, L1=0.0575, LPIPS=0.153]   


## Synthesis pipeline training

In [ ]:
# Comprehensive Counterfactual Training
from importlib import reload
from src.synthesis import train_comprehensive as train_comprehensive_module
train_comprehensive_module = reload(train_comprehensive_module)
from src.synthesis.train_comprehensive import ComprehensiveTrainer

cfg.sharpened_decoder_path = "outputs/synth_network/stylegan_decoder_sharpened/sharp_epoch_10.pth"

trainer = ComprehensiveTrainer(cfg, experiment_name="ceGAN_counterfactual")
trainer.train(
    num_epochs=cfg.num_epochs,
    loader_train=loader_train,
    loader_val=loader_val
)

📁 Experiment directory: outputs/synth_network/CF_generator/ceGAN_counterfactual_20251213
🔄 Loading Sharpened StyleGAN decoder from outputs/synth_network/stylegan_decoder_sharpened/sharp_epoch_10.pth...
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


/mnt/a/Ubuntu/lab.venv/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/mnt/a/Ubuntu/lab.venv/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /mnt/a/Ubuntu/lab.venv/lib/python3.11/site-packages/lpips/weights/v0.1/vgg.pth
🧭 Loaded decoder weights from outputs/synth_network/stylegan_decoder/latest.pth
🚀 Starting Comprehensive Counterfactual Training...
📊 Logging to: outputs/synth_network/CF_generator/ceGAN_counterfactual_20251213/logs
💾 Checkpoints to: outputs/synth_network/CF_generator/ceGAN_counterfactual_20251213/checkpoints
🎨 Visualizations to: outputs/synth_network/CF_generator/ceGAN_counterfactual_20251213/visualizations


Epoch 0 / 20: 100%|██████████| 4343/4343 [24:11<00:00,  2.99it/s, loss=0.4779, cf=0.023, ret=0.060]



Epoch 0 Summary
  total: 1.3208
  cf: 0.0647
  retention: 0.0364
  latent_prox: 0.0163
  ortho: 0.0916
  sparse: 0.0794
  5_o_Clock_Shadow: 0.0978
  Arched_Eyebrows: 0.0779
  Attractive: 0.0614
  Bags_Under_Eyes: 0.0514
  Bald: 0.0406
  Bangs: 0.0933
  Big_Lips: 0.0511
  Big_Nose: 0.0671
  Black_Hair: 0.0739
  Blond_Hair: 0.0892
  Blurry: 0.0246
  Brown_Hair: 0.0798
  Bushy_Eyebrows: 0.0847
  Chubby: 0.0499
  Double_Chin: 0.0865
  Eyeglasses: 0.0768
  Goatee: 0.0482
  Gray_Hair: 0.0606
  Heavy_Makeup: 0.0781
  High_Cheekbones: 0.0662
  Male: 0.0498
  Mouth_Slightly_Open: 0.0322
  Mustache: 0.0688
  Narrow_Eyes: 0.0273
  No_Beard: 0.0294
  Oval_Face: 0.0611
  Pale_Skin: 0.0653
  Pointy_Nose: 0.0680
  Receding_Hairline: 0.0543
  Rosy_Cheeks: 0.1138
  Sideburns: 0.0938
  Smiling: 0.0627
  Straight_Hair: 0.0766
  Stubble: 0.0741
  Sunglass: 0.0946
  Sweating: 0.0463
  Thick_Lips: 0.0660
  Thin_Lips: 0.0954
  Wearing_Earrings: 0.0747
  Young: 0.0412
✨ Best checkpoint saved: outputs/synth_n

Epoch 1 / 20: 100%|██████████| 4343/4343 [24:16<00:00,  2.98it/s, loss=0.4813, cf=0.023, ret=0.051]



Epoch 1 Summary
  total: 1.1765
  cf: 0.0575
  retention: 0.0376
  latent_prox: 0.0165
  ortho: 0.0863
  sparse: 0.0792
  5_o_Clock_Shadow: 0.0953
  Arched_Eyebrows: 0.0698
  Attractive: 0.0604
  Bags_Under_Eyes: 0.0384
  Bald: 0.0202
  Bangs: 0.0936
  Big_Lips: 0.0370
  Big_Nose: 0.0399
  Black_Hair: 0.0440
  Blond_Hair: 0.0621
  Blurry: 0.0203
  Brown_Hair: 0.0719
  Bushy_Eyebrows: 0.0543
  Chubby: 0.0451
  Double_Chin: 0.0758
  Eyeglasses: 0.0692
  Goatee: 0.0265
  Gray_Hair: 0.0478
  Heavy_Makeup: 0.0782
  High_Cheekbones: 0.0666
  Male: 0.0469
  Mouth_Slightly_Open: 0.0264
  Mustache: 0.0588
  Narrow_Eyes: 0.0241
  No_Beard: 0.0228
  Oval_Face: 0.0580
  Pale_Skin: 0.0550
  Pointy_Nose: 0.0609
  Receding_Hairline: 0.0524
  Rosy_Cheeks: 0.1089
  Sideburns: 0.0737
  Smiling: 0.0582
  Straight_Hair: 0.0689
  Stubble: 0.0679
  Sunglass: 0.0852
  Sweating: 0.0264
  Thick_Lips: 0.0668
  Thin_Lips: 0.0869
  Wearing_Earrings: 0.0513
  Young: 0.0390
✨ Best checkpoint saved: outputs/synth_n

Epoch 2 / 20:  50%|█████     | 2172/4343 [11:46<11:52,  3.04it/s, loss=1.7951, cf=0.089, ret=0.036]